In [1]:
import re
from pathlib import Path

# ---- inputs ----
input_counts = "GSE225221_cfrna_counts_CPM_GeneNames.txt"
out_dir = Path("Decon-COO-Results_Loy_COVID-MIS-C_nuSVR")

# ---- 1) read header samples (in order) ----
with open(input_counts, "r") as f:
    header = f.readline().rstrip("\n")

# split on tabs or whitespace (covers weird formatting)
tokens = re.split(r"[\t ]+", header)

# first token is gene_name; the rest are samples
header_samples = tokens[1:]

# ---- 2) read completed sample IDs from filenames ----
done_samples = set()
for p in out_dir.glob("nuSVR_CountsRMSE_Random_*.txt"):
    name = p.name
    # strip prefix/suffix
    sample = name.replace("nuSVR_CountsRMSE_Random_", "").replace(".txt", "")
    done_samples.add(sample)

# ---- 3) compute missing ----
missing = [s for s in header_samples if s not in done_samples]

print(f"Header samples: {len(header_samples)}")
print(f"Done samples (from filenames): {len(done_samples)}")
print(f"Missing samples: {len(missing)}\n")

# ---- 4) print missing with indices ----
print("Missing samples with indices:")
for s in missing:
    idx0 = header_samples.index(s)      # 0-based
    idx1 = idx0 + 1                     # 1-based
    print(f"{s}\tidx0={idx0}\tidx1={idx1}")

# Optional: write them to a file
with open("missing_samples.tsv", "w") as out:
    out.write("sample\tidx0\tidx1\n")
    for s in missing:
        idx0 = header_samples.index(s)
        out.write(f"{s}\t{idx0}\t{idx0+1}\n")
print("\nWrote missing_samples.tsv")


Header samples: 132
Done samples (from filenames): 131
Missing samples: 6

Missing samples with indices:
prevail_cu_cfrna_199	idx0=6	idx1=7
prevail_cu_cfrna_125	idx0=66	idx1=67
prevail_cu_cfrna_129	idx0=67	idx1=68
prevail_cu_cfrna_128	idx0=80	idx1=81
prevail_cu_cfrna_121	idx0=81	idx1=82
prevail_cu_cfrna_148	idx0=101	idx1=102

Wrote missing_samples.tsv


In [4]:
import pandas as pd

data = pd.read_csv(
    "GSE225221_cfrna_counts_CPM_GeneNames.txt",
    sep="\t",
    index_col=0
)

sample_names = data.columns

indices = [6, 66, 67, 80, 81, 101]
for i in indices:
    print(i, sample_names[i])


6 prevail_cu_cfrna_199
66 prevail_cu_cfrna_125
67 prevail_cu_cfrna_129
80 prevail_cu_cfrna_128
81 prevail_cu_cfrna_121
101 prevail_cu_cfrna_148


This script scans each directory for nuSVR_Rerun_*.txt files generated during the nuSVR hyperparameter search. For every sample, it reads the RMSE-PredictedCounts metric, identifies the C and nu combination that yields the lowest RMSE, and copies that file as the final output (nuSVR_CountsRMSE_Random_<sample>.txt). This replicates the selection logic of the slower original script but for the problematic samples.

In [1]:
import os
import glob
import pandas as pd
import shutil

# directory with rerun files
dir_pattern = "Decon-COO-Results_Loy_COVID-MIS-C_nuSVR/"
directories = sorted(glob.glob(dir_pattern))

print(f"Found {len(directories)} directories.")

for input_dir in directories:
    print(f"\nProcessing directory: {input_dir}")

    pattern = os.path.join(input_dir, "nuSVR_Rerun_*.txt")
    files = glob.glob(pattern)

    if not files:
        print("  No rerun files found — skipping.")
        continue

    samples = {}

    for f in files:
        name = os.path.basename(f)
        parts = name.replace(".txt", "").split("_")

        # sample ID = everything between 3rd token and C/nu
        sample_id = "_".join(parts[3:-2])

        # C and nu always at the end
        C_val = parts[-2].replace("C", "")
        nu_val = parts[-1].replace("nu", "")

        df = pd.read_csv(f, sep="\t")
        rmse = float(df["RMSE-PredictedCounts"].iloc[0])

        # keep best RMSE per sample
        if sample_id not in samples or rmse < samples[sample_id]["rmse"]:
            samples[sample_id] = {
                "rmse": rmse,
                "file": f,
                "C": C_val,
                "nu": nu_val
            }

    # copy and rename winning files
    for sample_id, info in samples.items():
        best_file = info["file"]
        rmse = info["rmse"]
        C_val = info["C"]
        nu_val = info["nu"]

        # output format you requested
        new_name = f"nuSVR_CountsRMSE_Random_{sample_id}.txt"
        new_path = os.path.join(input_dir, new_name)

        shutil.copy(best_file, new_path)

        print(f"  Sample: {sample_id}")
        print(f"    → Best RMSE: {rmse:.6f}")
        print(f"    → Best C:  {C_val}")
        print(f"    → Best nu: {nu_val}")
        print(f"    → Saved: {new_path}")


Found 1 directories.

Processing directory: Decon-COO-Results_Loy_COVID-MIS-C_nuSVR/
  Sample: cu_cfrna_129
    → Best RMSE: 0.994986
    → Best C:  10
    → Best nu: 0.05
    → Saved: Decon-COO-Results_Loy_COVID-MIS-C_nuSVR/nuSVR_CountsRMSE_Random_cu_cfrna_129.txt
  Sample: cu_cfrna_128
    → Best RMSE: 1.012485
    → Best C:  0.1
    → Best nu: 0.05
    → Saved: Decon-COO-Results_Loy_COVID-MIS-C_nuSVR/nuSVR_CountsRMSE_Random_cu_cfrna_128.txt
  Sample: cu_cfrna_125
    → Best RMSE: 0.993175
    → Best C:  10
    → Best nu: 0.05
    → Saved: Decon-COO-Results_Loy_COVID-MIS-C_nuSVR/nuSVR_CountsRMSE_Random_cu_cfrna_125.txt
  Sample: cu_cfrna_148
    → Best RMSE: 0.933632
    → Best C:  10
    → Best nu: 0.1
    → Saved: Decon-COO-Results_Loy_COVID-MIS-C_nuSVR/nuSVR_CountsRMSE_Random_cu_cfrna_148.txt
  Sample: cu_cfrna_199
    → Best RMSE: 0.863368
    → Best C:  0.1
    → Best nu: 0.75
    → Saved: Decon-COO-Results_Loy_COVID-MIS-C_nuSVR/nuSVR_CountsRMSE_Random_cu_cfrna_199.txt
  Sample: